In [2]:
from google.colab import drive
import os
import json
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report

# ==========================================
# STEP 0: CONNECT DRIVE & UNZIP
# ==========================================
drive.mount('/content/drive')
!unzip -q "/content/drive/MyDrive/archive (3).zip" -d "/content/dataset"
print("Extraction complete!")

# ==========================================
# STEP 1: AUTO-FIND THE CORRECT FOLDER
# ==========================================
base_path = '/content/dataset'
data_dir = None

for root, dirs, files in os.walk(base_path):
    if len(dirs) > 5:
        data_dir = root
        break

if data_dir is None:
    print("Error: Could not find the disease subfolders. Check your zip file.")
else:
    print(f"Success! Found dataset master folder at: {data_dir}")

# ==========================================
# STEP 2: PREP THE DATA
# ==========================================
batch_size = 32
img_height = 256
img_width = 256

print("Loading training data...")
train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)

print("Loading validation data...")
val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)

normalization_layer = tf.keras.layers.Rescaling(1./255)
normalized_train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
normalized_val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

# Get the class names and number of classes BEFORE building the model
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"\nAwesome! Found {num_classes} distinct categories of leaves.")

# ==========================================
# STEP 3: BUILD AND TRAIN THE CNN (UPDATED V2)
# ==========================================
model = tf.keras.Sequential([
  tf.keras.layers.RandomFlip("horizontal_and_vertical", input_shape=(img_height, img_width, 3)),
  tf.keras.layers.RandomRotation(0.2),

  tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2, 2),

  tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2, 2),

  tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2, 2),

  tf.keras.layers.Flatten(),
  tf.keras.layers.Dropout(0.5),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
  optimizer='adam',
  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
  metrics=['accuracy']
)

print("\nStarting training on Google's GPU...")
history = model.fit(
  normalized_train_ds,
  validation_data=normalized_val_ds,
  epochs=10
)

print("\nTraining Complete! You successfully built your module.")

# ==========================================
# STEP 4: EXPORT MODEL & CLASS NAMES
# ==========================================
model_path = '/content/drive/MyDrive/farmerhub_disease_model.keras'
model.save(model_path)
print(f"Model successfully saved to: {model_path}")

with open('/content/drive/MyDrive/class_names.json', 'w') as f:
    json.dump(class_names, f)
print("class_names.json saved successfully to your Google Drive!")

# ==========================================
# STEP 5: PERFORMANCE METRICS (From Corrections.pdf)
# ==========================================
y_true = []
y_pred = []

print("\nRunning final predictions on the validation set for the report...")
for images, labels in normalized_val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print("\n=== FINAL CLASSIFICATION REPORT ===")
print(classification_report(y_true, y_pred, target_names=class_names))

Mounted at /content/drive
Extraction complete!
Success! Found dataset master folder at: /content/dataset/PlantVillage/train
Loading training data...
Found 43444 files belonging to 38 classes.
Using 34756 files for training.
Loading validation data...
Found 43444 files belonging to 38 classes.
Using 8688 files for validation.

Awesome! Found 38 distinct categories of leaves.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Starting training on Google's GPU...
Epoch 1/10
1087/1087 ━━━━━━━━━━━━━━━━━━━━ 92s 77ms/step - accuracy: 0.6000 - loss: 1.4011 - val_accuracy: 0.7791 - val_loss: 0.7277
Epoch 2/10
1087/1087 ━━━━━━━━━━━━━━━━━━━━ 85s 78ms/step - accuracy: 0.7986 - loss: 0.6414 - val_accuracy: 0.8381 - val_loss: 0.5170
Epoch 3/10
1087/1087 ━━━━━━━━━━━━━━━━━━━━ 87s 80ms/step - accuracy: 0.8539 - loss: 0.4636 - val_accuracy: 0.7747 - val_loss: 0.7780
Epoch 4/10
1087/1087 ━━━━━━━━━━━━━━━━━━━━ 86s 79ms/step - accuracy: 0.8769 - loss: 0.3856 - val_accuracy: 0.8713 - val_loss: 0.4059
Epoch 5/10
1087/1087 ━━━━━━━━━━━━━━━━━━━━ 86s 79ms/step - accuracy: 0.8923 - loss: 0.3284 - val_accuracy: 0.8638 - val_loss: 0.4579
Epoch 6/10
1087/1087 ━━━━━━━━━━━━━━━━━━━━ 85s 78ms/step - accuracy: 0.9021 - loss: 0.3026 - val_accuracy: 0.8887 - val_loss: 0.3704
Epoch 7/10
1087/1087 ━━━━━━━━━━━━━━━━━━━━ 85s 78ms/step - accuracy: 0.9139 - loss: 0.2683 - val_accuracy: 0.9042 - val_loss: 0.3141
Epoch 8/10
1087/1087 ━━━━━━━━━━━━━━━━━